In [ ]:
import os
import json
import random
import hashlib
from typing import Dict, List, Optional

from datasets import load_dataset
from transformers import AutoTokenizer

# =========================================================
# Config
# =========================================================
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
OUT_DIR = "/root/data/instruct_data"
SEED = 0

N_SFT = 30_000
N_CALIB = 1_000

USE_HF_DATASET = True
HF_DATASET_ID = "HuggingFaceH4/ultrachat_200k"
HF_SPLIT = "train_sft"

# ---------- 长度约束 ----------
# SFT：偏训练友好
SFT_MIN_TOTAL_TOK = 64
SFT_MAX_TOTAL_TOK = 1024
SFT_MIN_TARGET_TOK = 16
SFT_MAX_TARGET_TOK = 512

# calib：覆盖更宽一点
CALIB_MIN_TOTAL_TOK = 64
CALIB_MAX_TOTAL_TOK = 1536
CALIB_MIN_TARGET_TOK = 8
CALIB_MAX_TARGET_TOK = 768

CALIB_BUCKETS = {
    "short": {"min": 64, "max": 256, "target": 300},
    "mid":   {"min": 257, "max": 768, "target": 500},
    "long":  {"min": 769, "max": 1536, "target": 200},
}
assert sum(v["target"] for v in CALIB_BUCKETS.values()) == N_CALIB

# 最多保留多少条 message，避免极端长多轮对话
MAX_MESSAGES_PER_SAMPLE = 12

In [ ]:
# =========================================================
# Utils
# =========================================================
def set_seed(seed: int):
    random.seed(seed)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def write_jsonl(path: str, records: List[Dict]):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def sha1_text(x: str) -> str:
    return hashlib.sha1(x.encode("utf-8")).hexdigest()

def normalize_text(x: Optional[str]) -> str:
    if x is None:
        return ""
    return str(x).strip()

def normalize_messages(messages: List[Dict]) -> List[Dict]:
    out = []
    for m in messages:
        if not isinstance(m, dict):
            continue
        role = m.get("role")
        content = m.get("content")
        if role in {"system", "user", "assistant"} and isinstance(content, str):
            content = content.strip()
            if content:
                out.append({"role": role, "content": content})
    return out

def truncate_messages(messages: List[Dict], max_messages: int) -> List[Dict]:
    """
    保留最后若干条消息，但尽量保留开头 system。
    """
    if len(messages) <= max_messages:
        return messages

    if messages and messages[0]["role"] == "system":
        sys_msg = [messages[0]]
        rest = messages[1:]
        keep_rest = rest[-(max_messages - 1):]
        return sys_msg + keep_rest
    return messages[-max_messages:]

def get_text_len(tok, text: str) -> int:
    return len(tok(text, add_special_tokens=False)["input_ids"])

def get_chat_len(tok, messages: List[Dict], add_generation_prompt: bool) -> int:
    text = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt
    )
    return len(tok(text, add_special_tokens=False)["input_ids"])

def load_ultrachat_examples():
    if USE_HF_DATASET:
        ds = load_dataset(HF_DATASET_ID, split=HF_SPLIT)
        return ds
    raise ValueError("Please enable USE_HF_DATASET.")

def extract_training_examples(messages: List[Dict]) -> List[Dict]:
    """
    把多轮对话展开成多个监督样本：
      prompt_messages -> target_assistant

    输出格式：
    {
      "prompt_messages": [...],
      "target_text": "...",
      "messages": prompt + assistant
    }
    """
    msgs = normalize_messages(messages)
    msgs = truncate_messages(msgs, MAX_MESSAGES_PER_SAMPLE)

    examples = []
    history = []

    for m in msgs:
        if m["role"] == "assistant":
            if any(x["role"] == "user" for x in history):
                prompt_messages = [dict(x) for x in history]
                full_messages = prompt_messages + [dict(m)]
                examples.append({
                    "prompt_messages": prompt_messages,
                    "target_text": m["content"],
                    "messages": full_messages,
                })
            history.append(m)
        else:
            history.append(m)

    return examples

def build_record(ex_obj: Dict, tok, source_idx: int, turn_idx: int) -> Optional[Dict]:
    messages = ex_obj["messages"]
    prompt_messages = ex_obj["prompt_messages"]
    target_text = ex_obj["target_text"]

    if not messages or not prompt_messages or not target_text:
        return None

    prompt_len = get_chat_len(tok, prompt_messages, add_generation_prompt=True)
    total_len = get_chat_len(tok, messages, add_generation_prompt=False)
    target_len = get_text_len(tok, target_text)

    # 用最后一个 user + target 做轻量去重
    last_user = ""
    for m in reversed(prompt_messages):
        if m["role"] == "user":
            last_user = m["content"]
            break

    uid = sha1_text(
        f"{source_idx}|||{turn_idx}|||{last_user[:300]}|||{target_text[:300]}"
    )

    return {
        "id": uid,
        "messages": messages,
        "prompt_messages": prompt_messages,
        "target_text": target_text,
        "meta": {
            "source_index": source_idx,
            "turn_index": turn_idx,
            "prompt_tokens": prompt_len,
            "target_tokens": target_len,
            "total_tokens": total_len,
            "num_messages": len(messages),
        }
    }

def sft_accept(rec: Dict) -> bool:
    m = rec["meta"]
    return (
        SFT_MIN_TOTAL_TOK <= m["total_tokens"] <= SFT_MAX_TOTAL_TOK and
        SFT_MIN_TARGET_TOK <= m["target_tokens"] <= SFT_MAX_TARGET_TOK
    )

def calib_bucket_name(total_tokens: int) -> Optional[str]:
    for name, cfg in CALIB_BUCKETS.items():
        if cfg["min"] <= total_tokens <= cfg["max"]:
            return name
    return None

def calib_accept(rec: Dict) -> Optional[str]:
    m = rec["meta"]
    if not (
        CALIB_MIN_TOTAL_TOK <= m["total_tokens"] <= CALIB_MAX_TOTAL_TOK and
        CALIB_MIN_TARGET_TOK <= m["target_tokens"] <= CALIB_MAX_TARGET_TOK
    ):
        return None
    return calib_bucket_name(m["total_tokens"])

In [ ]:
# =========================================================
# Main
# =========================================================
set_seed(SEED)
ensure_dir(OUT_DIR)

print(f"Loading tokenizer: {MODEL_NAME}")
tok = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True
)

print("Loading UltraChat data...")
examples = load_ultrachat_examples()
print(f"Loaded examples: {len(examples)}")

idxs = list(range(len(examples)))
random.shuffle(idxs)

sft_records: List[Dict] = []
calib_buckets: Dict[str, List[Dict]] = {k: [] for k in CALIB_BUCKETS}

seen_sft = set()
seen_calib = set()

def calib_done() -> bool:
    return all(len(calib_buckets[k]) >= CALIB_BUCKETS[k]["target"] for k in CALIB_BUCKETS)

for n_scanned, i in enumerate(idxs, start=1):
    raw = examples[i]
    msgs = raw.get("messages", None)
    if not isinstance(msgs, list):
        continue

    expanded = extract_training_examples(msgs)
    if not expanded:
        continue

    for turn_idx, ex_obj in enumerate(expanded):
        rec = build_record(ex_obj, tok, source_idx=i, turn_idx=turn_idx)
        if rec is None:
            continue

        dup_key = sha1_text(
            rec["prompt_messages"][-1]["content"][:500] + "|||" + rec["target_text"][:500]
            if rec["prompt_messages"] else rec["target_text"][:500]
        )

        # ---------- SFT ----------
        if len(sft_records) < N_SFT and dup_key not in seen_sft and sft_accept(rec):
            sft_records.append(rec)
            seen_sft.add(dup_key)

        # ---------- calib ----------
        bucket = calib_accept(rec)
        if bucket is not None:
            if (
                len(calib_buckets[bucket]) < CALIB_BUCKETS[bucket]["target"]
                and dup_key not in seen_calib
            ):
                calib_buckets[bucket].append(rec)
                seen_calib.add(dup_key)

        if len(sft_records) >= N_SFT and calib_done():
            break

    if len(sft_records) >= N_SFT and calib_done():
        break

    if n_scanned % 10000 == 0:
        calib_count = sum(len(v) for v in calib_buckets.values())
        print(
            f"scanned={n_scanned} | "
            f"sft={len(sft_records)}/{N_SFT} | "
            f"calib={calib_count}/{N_CALIB} | "
            f"buckets={{"
            + ", ".join(f'{k}:{len(v)}/{CALIB_BUCKETS[k]["target"]}' for k, v in calib_buckets.items())
            + "}"
        )

calib_records = []
for k in ["short", "mid", "long"]:
    calib_records.extend(calib_buckets[k])

if len(sft_records) < N_SFT:
    raise RuntimeError(
        f"Not enough SFT records: got {len(sft_records)}, need {N_SFT}. "
        f"Try loosening SFT_MAX_TOTAL_TOK / SFT_MAX_TARGET_TOK."
    )

if len(calib_records) < N_CALIB:
    raise RuntimeError(
        f"Not enough calib records: got {len(calib_records)}, need {N_CALIB}. "
        f"Try loosening CALIB_MAX_TOTAL_TOK / CALIB_MAX_TARGET_TOK."
    )

random.shuffle(sft_records)
random.shuffle(calib_records)

sft_path = os.path.join(OUT_DIR, f"sft_{N_SFT}.jsonl")
calib_path = os.path.join(OUT_DIR, f"calib_{N_CALIB}.jsonl")

write_jsonl(sft_path, sft_records[:N_SFT])
write_jsonl(calib_path, calib_records[:N_CALIB])

print("\nDone.")
print("SFT   :", sft_path, len(sft_records[:N_SFT]))
print("Calib :", calib_path, len(calib_records[:N_CALIB]))
print("Calib bucket counts:", {k: len(v) for k, v in calib_buckets.items()})

In [ ]:
def summarize_lengths(records: List[Dict], name: str):
    totals = sorted(r["meta"]["total_tokens"] for r in records)
    targets = sorted(r["meta"]["target_tokens"] for r in records)

    def pct(arr, p):
        idx = min(len(arr) - 1, int(len(arr) * p))
        return arr[idx]

    print(f"\n{name} length summary:")
    print(f"  total_tokens  p50={pct(totals, 0.50)} p90={pct(totals, 0.90)} p95={pct(totals, 0.95)} max={totals[-1]}")
    print(f"  target_tokens p50={pct(targets, 0.50)} p90={pct(targets, 0.90)} p95={pct(targets, 0.95)} max={targets[-1]}")

summarize_lengths(sft_records[:N_SFT], "SFT")
summarize_lengths(calib_records[:N_CALIB], "Calib")

print("\nSample SFT record:")
print(json.dumps(sft_records[0], ensure_ascii=False, indent=2)[:1200])

print("\nSample Calib record:")
print(json.dumps(calib_records[0], ensure_ascii=False, indent=2)[:1200])